In [1]:
import os

In [2]:
%pwd

'd:\\Project\\chicken-disease-classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Project\\chicken-disease-classification'

In [ ]:
#data ingestion related entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [9]:
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config



    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")

        else:
            logger.info(f"File already exists of size :{get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        zip_file_path : str
        Extract the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-03-01 11:48:45,888]: INFO: common: yaml file: config\config.yaml loaded successfully
[2026-03-01 11:48:45,889]: INFO: common: yaml file: params.yaml loaded successfully
[2026-03-01 11:48:45,889]: INFO: common: created directory at: artifacts
[2026-03-01 11:48:45,891]: INFO: common: created directory at: artifacts/data_ingestion
[2026-03-01 11:48:47,611]: INFO: 2832211773: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 11634040
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "3a42ee6026c16f619ed41fa56a4f048aa872271208877a75335d3425d38bea19"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: D25C:159E1D:3B4060:42EFFF:69A427A1
Accept-Ranges: bytes
Date: Sun, 01 Mar 2026 11:48:50 GMT
Via: 1.1 varnish
X-Served-By: cache-ams21038-AMS
X-Cache: